In [1]:
import wandb
import pandas as pd
import numpy as np

import json

api = wandb.Api()

In [2]:
def linearize_dict(d, parent_key='', sep='.'):
    """
    Linearizes a nested dictionary into a flat dictionary with keys in the format `key1.key2`.

    :param d: The dictionary to linearize.
    :param parent_key: The current key being processed (used for recursion).
    :param sep: The separator between keys.
    :return: A flattened dictionary.
    """
    items = []
    
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.extend(linearize_dict(v, new_key, sep).items())
        else:
            items.append((new_key, v))
    
    return dict(items)

def get_runs_df(runs):
    runs_list = []
    for run in runs: 
        runs_list.append({
            **run.summary._json_dict,
            **linearize_dict({k: v for k,v in run.config.items()
            if not k.startswith('_')}),
            **linearize_dict(run.metadata if run.metadata else {}),
            **{"name": run.name, "id": run.id}
            })

    runs_df = pd.DataFrame(runs_list)
    return runs_df

In [3]:
iauc_dauc_runs = api.runs(
    "pasqualedem/affex",
    filters={
        "$and": [
            {"config.metric.n_steps": 75},
            {"tags": {"$nin": ["Broken"]}},
            {"$or": [
                {'config.dataset.datasets.val_deepglobe_N1K5.name': "deepglobe"},
                {'config.dataset.datasets.val_lung_N1K5.name': "lung"},
                {'config.dataset.datasets.val_isic_N1K5.name': "isic"},
            ]}
        ]
    }
)

In [4]:
iauc_dauc_runs_df = get_runs_df(iauc_dauc_runs)

In [41]:
res_df = iauc_dauc_runs_df.copy()

dataset_names = ["val_lung_N1K5", "val_isic_N1K5", "val_deepglobe_N1K5"]

dataset_cols = [f'dataset.datasets.{name}.name' for name in dataset_names]
shots_cols = [f'dataset.datasets.{name}.n_shots' for name in dataset_names]
res_df["Shots"] = res_df[shots_cols].fillna("").astype(str).sum(axis=1).astype(float).astype(int)
res_df["Dataset"] = res_df[dataset_cols].fillna("").sum(axis=1) 

iauc_cols = [f"{name}_iauc" for name in dataset_names]
dauc_cols = [f"{name}_dauc" for name in dataset_names]
assert res_df[iauc_cols].isna().any(axis=1).all(), "More than one dataset is present in the same run, please check the dataset columns."
assert res_df[dauc_cols].isna().any(axis=1).all(), "More than one dataset is present in the same run, please check the dataset columns."


renamings_dict = {
    "explainer.name": "Explanation Method",
    "config.metric.n_steps": "N. Steps",
    "config.metric.iauc": "IAUC",
    "config.metric.dauc": "DAUC",
    "model": "Model",
}
res_df = res_df.rename(columns=renamings_dict)

value_renamings_dict = {
    "integrated_gradients": "Integrated Gradients",
    "saliency": "Saliency",
    "affinity": "Affinity Explainer (ours)",
    "random": "Random",
    "pascal": "Pascal 5^i",
    "coco": "COCO 20^i",
    "dcama": "DCAMA",
    "dmtnet": "DMTNet",
    "isic": "ISIC",
    "lung": "Lung",
    "deepglobe": "Deepglobe",
}
res_df["Explanation Method"] = res_df["Explanation Method"].replace(value_renamings_dict)
res_df["Dataset"] = res_df["Dataset"].replace(value_renamings_dict)
res_df["Model"] = res_df["Model"].replace(value_renamings_dict)


res_df["IAUC"] = res_df[iauc_cols].fillna(0).sum(axis=1)
res_df["DAUC"] = res_df[dauc_cols].fillna(0).sum(axis=1)

res_df["Diff."] = res_df["IAUC"] - res_df["DAUC"]

cols = ["Model", "Explanation Method", "Dataset", "Shots"]
res_df = res_df[cols + ["IAUC", "DAUC", "Diff."]]

# Define custom sort order
dataset_order = {"Deepglobe": 0, "ISIC": 1, "Lung": 2}
metric_order = {'IAUC': 0, 'DAUC': 1, 'Diff.': 2}
method_order = {
    'Random': 0,
    'Integrated Gradients': 1,
    'Saliency': 2,
    'Affinity Explainer': 3,
}
model_order = {
    'DCAMA': 0,
    'DMTNet': 1,
}

print(res_df)

res_df = res_df.pivot_table(
    index=['Dataset', 'Explanation Method'],
    columns=["Shots"],
    values=['IAUC', 'DAUC', 'Diff.']
).sort_index(axis=1).swaplevel(0, 1, axis=1)


new_cols  = sorted(res_df.columns, key=lambda x: (x[0], metric_order[x[1]]))
res_df = res_df[new_cols]
# Sort by Model and then by Explanation Method using the provided orderings
res_df = res_df.sort_values(
    by=['Dataset', 'Explanation Method'],  # this ensures level names exist and are used
    key=lambda x: x.map(dataset_order) if x.name == 'Dataset' else x.map(method_order)
)

iauc_dauc_df = (res_df*100).round(2)

iauc_dauc_df

     Model         Explanation Method    Dataset  Shots      IAUC      DAUC  \
0   DMTNet                   Saliency       ISIC      5  0.732147  0.781361   
1   DMTNet       Integrated Gradients       ISIC      5  0.653066  0.730476   
2   DMTNet  Affinity Explainer (ours)       ISIC      5  0.790059  0.644435   
3   DMTNet                     Random       ISIC      5  0.739172  0.739680   
4   DMTNet                     Random       ISIC      5  0.739172  0.739680   
5   DMTNet                   Saliency       Lung      5  0.812691  0.746967   
6   DMTNet       Integrated Gradients       Lung      5  0.719256  0.748969   
7   DMTNet  Affinity Explainer (ours)       Lung      5  0.902064  0.643075   
8   DMTNet                   Saliency  Deepglobe      5  0.461447  0.558001   
9   DMTNet                     Random       Lung      5  0.732055  0.732375   
10  DMTNet       Integrated Gradients  Deepglobe      5  0.245286  0.375693   
11  DMTNet  Affinity Explainer (ours)  Deepglobe    

Shots                                    5              
                                      IAUC   DAUC  Diff.
Dataset   Explanation Method                            
Deepglobe Random                     30.92  30.98  -0.05
          Integrated Gradients       24.53  37.57 -13.04
          Saliency                   46.14  55.80  -9.66
          Affinity Explainer (ours)  53.12  38.63  14.49
ISIC      Random                     73.92  73.97  -0.05
          Integrated Gradients       65.31  73.05  -7.74
          Saliency                   73.21  78.14  -4.92
          Affinity Explainer (ours)  79.01  64.44  14.56
Lung      Random                     73.21  73.24  -0.03
          Integrated Gradients       71.93  74.90  -2.97
          Saliency                   81.27  74.70   6.57
          Affinity Explainer (ours)  90.21  64.31  25.90

In [6]:
from tabulate import tabulate

latex_tab = tabulate(res_df, headers='keys', tablefmt='latex_booktabs', showindex=True)
print(latex_tab)

\begin{tabular}{lrrr}
\toprule
                                            &     IAUC &     DAUC &        Diff. \\
\midrule
 ('Deepglobe', 'Random')                    & 0.309229 & 0.309751 & -0.0005225   \\
 ('Deepglobe', 'Integrated Gradients')      & 0.245286 & 0.375693 & -0.130407    \\
 ('Deepglobe', 'Saliency')                  & 0.461447 & 0.558001 & -0.0965538   \\
 ('Deepglobe', 'Affinity Explainer (ours)') & 0.531181 & 0.386286 &  0.144894    \\
 ('ISIC', 'Random')                         & 0.739172 & 0.73968  & -0.000508215 \\
 ('ISIC', 'Integrated Gradients')           & 0.653066 & 0.730476 & -0.0774099   \\
 ('ISIC', 'Saliency')                       & 0.732147 & 0.781361 & -0.0492141   \\
 ('ISIC', 'Affinity Explainer (ours)')      & 0.790059 & 0.644435 &  0.145624    \\
 ('Lung', 'Random')                         & 0.732055 & 0.732375 & -0.000320113 \\
 ('Lung', 'Integrated Gradients')           & 0.719256 & 0.748969 & -0.0297134   \\
 ('Lung', 'Saliency')               

## IAUC %

In [13]:
iauc_p_runs = api.runs(
    "pasqualedem/affex",
    filters={
        "$and": [
            {"config.metric.percentage": {"$ne": None}},
            {"tags": {"$nin": ["Broken"]}},
            {"$or": [
                {'config.dataset.datasets.val_deepglobe_N1K5.name': "deepglobe"},
                {'config.dataset.datasets.val_lung_N1K5.name': "lung"},
                {'config.dataset.datasets.val_isic_N1K5.name': "isic"},
                {'config.dataset.datasets.val_deepglobe_N1K1.name': "deepglobe"},
                {'config.dataset.datasets.val_lung_N1K1.name': "lung"},
                {'config.dataset.datasets.val_isic_N1K1.name': "isic"},
            ]}
        ]
    }
)

In [14]:
iauc_p_runs_df = get_runs_df(iauc_p_runs)

In [42]:
res_df = iauc_p_runs_df.copy()

dataset_names = ["val_lung_N1K5", "val_isic_N1K5", "val_deepglobe_N1K5", "val_lung_N1K1", "val_isic_N1K1", "val_deepglobe_N1K1"]

dataset_cols = [f'dataset.datasets.{name}.name' for name in dataset_names]
shots_cols = [f'dataset.datasets.{name}.n_shots' for name in dataset_names]
res_df["Shots"] = res_df[shots_cols].fillna("").astype(str).sum(axis=1).astype(float).astype(int)
res_df["Dataset"] = res_df[dataset_cols].fillna("").sum(axis=1) 

iauc_cols = [f"{name}_iauc" for name in dataset_names]
assert res_df[iauc_cols].isna().any(axis=1).all(), "More than one dataset is present in the same run, please check the dataset columns."


renamings_dict = {
    "explainer.name": "Explanation Method",
    "config.metric.n_steps": "N. Steps",
    "config.metric.iauc": "IAUC",
    "config.metric.dauc": "DAUC",
    "model": "Model",
    "metric.percentage": "Percentage",
    "metric.loss": "Loss",
}
res_df = res_df.rename(columns=renamings_dict)

value_renamings_dict = {
    "integrated_gradients": "Integrated Gradients",
    "saliency": "Saliency",
    "affinity": "Affinity Explainer (ours)",
    "random": "Random",
    "pascal": "Pascal 5^i",
    "coco": "COCO 20^i",
    "dcama": "DCAMA",
    "dmtnet": "DMTNet",
    "isic": "ISIC",
    "lung": "Lung",
    "deepglobe": "Deepglobe",
}
res_df["Explanation Method"] = res_df["Explanation Method"].replace(value_renamings_dict)
res_df["Dataset"] = res_df["Dataset"].replace(value_renamings_dict)
res_df["Model"] = res_df["Model"].replace(value_renamings_dict)


res_df["iAUC"] = res_df[iauc_cols].fillna(0).sum(axis=1)
res_df["Loss"] = res_df["Loss"].fillna(False)

cols = ["Model", "Explanation Method", "Dataset", "Shots", "Percentage", "Loss"]
res_df = res_df[cols + ["iAUC"]]

# Define custom sort order
dataset_order = {"Deepglobe": 0, "ISIC": 1, "Lung": 2}
method_order = {
    'Random': 0,
    'Integrated Gradients': 1,
    'Saliency': 2,
    'Affinity Explainer': 3,
}
model_order = {
    'DCAMA': 0,
    'DMTNet': 1,
}

res_df = res_df.pivot_table(
    index=['Dataset', 'Explanation Method'],
    columns=['Shots', 'Percentage', "Loss"],
    values='iAUC'
).sort_index(axis=1)

perc_map = {
    False: "IAUC@",
    True: "mIoULoss@"
}

# Flatten the MultiIndex columns
# Collapse last two levels into one
res_df.columns = pd.MultiIndex.from_tuples([
    (lv0, f"{perc_map[lv2]}{lv1}") for lv0, lv1, lv2 in res_df.columns
])


# Sort by Model and then by Explanation Method using the provided orderings
res_df = res_df.sort_values(
    by=['Dataset', 'Explanation Method'],  # this ensures level names exist and are used
    key=lambda x: x.map(dataset_order) if x.name == 'Dataset' else x.map(method_order)
)

iauc_p_df = (res_df*100).round(2)

iauc_p_df

/tmp/ipykernel_548612/4046594982.py:44: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  res_df["Loss"] = res_df["Loss"].fillna(False)


1                       5  \
                                    mIoULoss@0.01 mIoULoss@0.05 IAUC@0.01   
Dataset   Explanation Method                                                
Deepglobe Random                            14.12         22.21     42.85   
          Integrated Gradients              18.19         24.93     42.22   
          Saliency                          15.02         21.51     44.21   
          Affinity Explainer (ours)         17.42         17.56     43.52   
ISIC      Random                            20.18         19.63     70.72   
          Integrated Gradients              15.64         13.48     75.36   
          Saliency                          15.29          8.48     79.63   
          Affinity Explainer (ours)         10.12          7.08     78.20   
Lung      Random                            31.22         51.90     62.89   
          Integrated Gradients              26.51         28.58     71.24   
          Saliency                          26.97         27.58     72.42   
          Affinity Explainer (ours)         26.36         28.32     79.59   

                                                                 
                                    mIoULoss@0.01 mIoULoss@0.05  
Dataset   Explanation Method                                     
Deepglobe Random                            14.89         27.60  
          Integrated Gradients              15.96         25.01  
          Saliency                          13.50         14.40  
          Affinity Explainer (ours)         12.94         13.10  
ISIC      Random                            22.49         18.71  
          Integrated Gradients              20.66         15.77  
          Saliency                          19.06         14.13  
          Affinity Explainer (ours)         18.08         12.47  
Lung      Random                            27.76         56.42  
          Integrated Gradients              25.91         22.00  
          Saliency                          25.96         24.89  
          Affinity Explainer (ours)         27.01         26.52

---

In [48]:
# joining iauc_dauc_df and iauc_p_df on index
final = iauc_dauc_df.join(iauc_p_df)
final = final.sort_index(level=0, axis=1)

metric_order = {
    'IAUC': 0,
    'DAUC': 1,
    'Diff.': 2,
    'IAUC@0.01': 3,
    'IAUC@0.05': 4,
    'mIoULoss@0.01': 5,
    'mIoULoss@0.05': 6,
}

new_cols  = sorted(final.columns, key=lambda x: (-x[0], metric_order[x[1]]))
final = final[new_cols]
final

5                          \
                                      IAUC   DAUC  Diff. IAUC@0.01   
Dataset   Explanation Method                                         
Deepglobe Random                     30.92  30.98  -0.05     42.85   
          Integrated Gradients       24.53  37.57 -13.04     42.22   
          Saliency                   46.14  55.80  -9.66     44.21   
          Affinity Explainer (ours)  53.12  38.63  14.49     43.52   
ISIC      Random                     73.92  73.97  -0.05     70.72   
          Integrated Gradients       65.31  73.05  -7.74     75.36   
          Saliency                   73.21  78.14  -4.92     79.63   
          Affinity Explainer (ours)  79.01  64.44  14.56     78.20   
Lung      Random                     73.21  73.24  -0.03     62.89   
          Integrated Gradients       71.93  74.90  -2.97     71.24   
          Saliency                   81.27  74.70   6.57     72.42   
          Affinity Explainer (ours)  90.21  64.31  25.90     79.59   

                                                                            1  \
                                    mIoULoss@0.01 mIoULoss@0.05 mIoULoss@0.01   
Dataset   Explanation Method                                                    
Deepglobe Random                            14.89         27.60         14.12   
          Integrated Gradients              15.96         25.01         18.19   
          Saliency                          13.50         14.40         15.02   
          Affinity Explainer (ours)         12.94         13.10         17.42   
ISIC      Random                            22.49         18.71         20.18   
          Integrated Gradients              20.66         15.77         15.64   
          Saliency                          19.06         14.13         15.29   
          Affinity Explainer (ours)         18.08         12.47         10.12   
Lung      Random                            27.76         56.42         31.22   
          Integrated Gradients              25.91         22.00         26.51   
          Saliency                          25.96         24.89         26.97   
          Affinity Explainer (ours)         27.01         26.52         26.36   

                                                   
                                    mIoULoss@0.05  
Dataset   Explanation Method                       
Deepglobe Random                            22.21  
          Integrated Gradients              24.93  
          Saliency                          21.51  
          Affinity Explainer (ours)         17.56  
ISIC      Random                            19.63  
          Integrated Gradients              13.48  
          Saliency                           8.48  
          Affinity Explainer (ours)          7.08  
Lung      Random                            51.90  
          Integrated Gradients              28.58  
          Saliency                          27.58  
          Affinity Explainer (ours)         28.32

In [49]:
# turn the two level index into two columns
final_to_latex = final.reset_index()

latex_tab = tabulate(final_to_latex, headers='keys', tablefmt='latex_booktabs', showindex=False)
print(latex_tab)

\begin{tabular}{llrrrrrrrr}
\toprule
 ('Dataset', '')   & ('Explanation Method', '')   &   (5, 'IAUC') &   (5, 'DAUC') &   (5, 'Diff.') &   (5, 'IAUC@0.01') &   (5, 'mIoULoss@0.01') &   (5, 'mIoULoss@0.05') &   (1, 'mIoULoss@0.01') &   (1, 'mIoULoss@0.05') \\
\midrule
 Deepglobe         & Random                       &         30.92 &         30.98 &          -0.05 &              42.85 &                  14.89 &                  27.6  &                  14.12 &                  22.21 \\
 Deepglobe         & Integrated Gradients         &         24.53 &         37.57 &         -13.04 &              42.22 &                  15.96 &                  25.01 &                  18.19 &                  24.93 \\
 Deepglobe         & Saliency                     &         46.14 &         55.8  &          -9.66 &              44.21 &                  13.5  &                  14.4  &                  15.02 &                  21.51 \\
 Deepglobe         & Affinity Explainer (ours)    &         53